In [ ]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%load_ext autoreload
%autoreload 2

import random
import numpy as np
import pandas as pd
import torch

import Python.config as cfg
import Python.utils as ut
import Python.simfun as sim
import Python.model2 as md
import Python.metric as me
import Python.bnn_mcmc as mcmc
import Python.bnn_train as train

seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

X, y, feature_true, signal, sim_info = sim.simfun_nonlinear(
    n=160,
    p=1,
    n_active=1,
    interaction=False,
    seed=seed,
    device=device,
)

In [ ]:
# -----------------------------
# 1. Split data and make grids
# -----------------------------

split_cfg = cfg.SplitConfig(
    train_frac=0.60,
    val_frac=0.20,
    test_frac=0.20,
    seed=seed,
)

indices = ut.make_split(X.shape[0], split_cfg)
splits = ut.split_data(X, y, indices, mode="tensor")
signal_splits = ut.split_data(X, signal, indices, mode="tensor")

X_train = splits["X_train"]
y_train = splits["y_train"]
X_test = splits["X_test"]
signal_test = signal_splits["y_test"]

x_val_grid = torch.linspace(
    -torch.pi, torch.pi, 201, device=device, dtype=X_train.dtype
)[:, None]
signal_val_grid = torch.cos(x_val_grid[:, 0])

x_grid = torch.linspace(
    -torch.pi, torch.pi, 400, device=device, dtype=X_train.dtype
)[:, None]
signal_grid = torch.cos(x_grid[:, 0])

In [ ]:
# -----------------------------------------
# 2. One-block edge model with mixed gates
# -----------------------------------------

arch = dict(
    input_dim=1,
    d_model=2,
    n_blocks=1,
    ffn_dims=3,
    out_dim=1,
    bounded=None,
    gate_power=2.0,
    gate_tau=1.0,
    sigmoid_params=("E", "Wout"),
    sigmoid_tau=1.0,
    attention_type="none",
    ffn_activation="relu",
)

mcmc_model = md.LaSTBNNVI(
    X=X_train,
    y=y_train,
    family=sim_info["family"],
    sigma2=sim_info["sigma2"],
    init_sd=0.5,
    K_flow=0,
    **arch,
).to(device)

xi_zero = torch.zeros(1, mcmc_model.decoder.dim, device=device)
_, gate_zero = mcmc_model.decoder.unpack(
    xi_zero, return_summary=True
)

assert torch.allclose(gate_zero["E_gate_mean"], torch.full_like(gate_zero["E_gate_mean"], 0.5))
assert torch.allclose(gate_zero["Wout_gate_mean"], torch.full_like(gate_zero["Wout_gate_mean"], 0.5))
assert torch.all(gate_zero["W1_0_gate_mean"] == 0)

print(gate_zero["gate_type_by_parameter"])

In [ ]:
# -----------------------------
# 3. Shared-likelihood MCMC
# -----------------------------

mcmc_ref = mcmc.run_bnn_mcmc(
    model=mcmc_model,
    N=10000,
    S_max=100,
    burnin=2000,
    thin=1,
    seed=seed,
    print_every=500,
)

mcmc_xi = torch.as_tensor(
    mcmc_ref["xi_draws"],
    device=device,
    dtype=X_train.dtype,
)

In [ ]:
# ------------------------------------
# 4. Train mixed-gate edge-level RaT
# ------------------------------------

out = train.train_edge_bnn(
    X_train=X_train,
    y_train=y_train,
    X_eval=x_val_grid,
    signal_eval=signal_val_grid,
    X_final=X_test,
    signal_final=signal_test,
    mcmc_decoder=mcmc_model.decoder,
    mcmc_xi=mcmc_xi,
    family=sim_info["family"],
    sigma2=sim_info["sigma2"],
    init_sd=0.5,
    K_flow=8,
    flow_hidden_units=64,
    flow_hidden_layers=2,
    scale_clip=1.5,
    epochs=6000,
    lr=3e-4,
    R_train=100,
    R_eval=1000,
    R_final=5000,
    eval_every=250,
    sigmoid_zero_threshold=0.05,
    seed=seed,
    **arch,
)

In [ ]:
# -----------------------------
# 5. Selected checkpoint
# -----------------------------

history = out["history"]
best_id = history["last_signal_r2"].idxmax()

print(
    history.loc[
        best_id,
        [
            "epoch",
            "expected_log_likelihood",
            "kl_q_prior",
            "last_signal_r2",
            "hidden_a_skl",
            "hidden_pip_rmse",
            "expected_functional_paths",
            "zero_functional_path_prob",
            "E_gate_mean",
            "Wout_gate_mean",
        ],
    ]
)

In [ ]:
from IPython.display import display

print("===== Overall metrics =====")
display(pd.DataFrame([out["final"]["summary"]]).T.rename(columns={0: "value"}))

print("\n===== Posterior recovery by edge group =====")
display(out["final"]["posterior_by_layer"])

print("\n===== Hidden-unit connectivity =====")
display(out["final"]["hidden_units"])

print("\n===== Effective path summary =====")
display(out["final"]["path_summary"])

print("\n===== Effective path by hidden unit =====")
display(out["final"]["path_units"])

In [ ]:
# Every checkpoint is retained.
display(out["history"])
display(out["layer_history"])
display(out["path_history"])

In [ ]:
# -----------------------------
# 6. Function posterior plot
# -----------------------------

mcmc_grid = me.predict_draws(
    mcmc_model.decoder, x_grid, mcmc_xi
)
rat_grid = me.predict_draws(
    out["model"].decoder, x_grid, out["final"]["xi"]
)

fig, ax = me.plot_function_1d(
    x=x_grid[:, 0].cpu(),
    signal=signal_grid.cpu(),
    mcmc_pred_draws=mcmc_grid,
    rat_pred_draws=rat_grid,
)

`sigmoid_zero_threshold=0.05` is used only in path diagnostics. Training, ELBO evaluation, and MCMC always use the continuous sigmoid values; no hard truncation is applied to the decoder.